# E6 - GUARDRAILS + LANGGRAPH + MULTI-AGENTES COM LANGCHAIN

**Objetivo:** Implementar e testar TODOS os topicos:
1. Guardrails (5 tipos)
2. LangGraph (grafo com nodes e conditional edges)
3. Multi-Agentes (supervisor + 3 agentes com LLM REAL)

**Duracao:** 90 minutos

**LLM:** Ollama local (gratuito) + OpenRouter (fallback)

**Requisitos:**
- Python 3.10+
- Ollama rodando (ou OpenRouter API key)
- LangChain instalado

## PARTE 0: SETUP E CONFIGURACAO

In [14]:
# Instalar dependencias
import subprocess
import sys

packages = ['langchain', 'langgraph', 'requests']

for package in packages:
    try:
        __import__(package)
        print(f"OK - {package} ja instalado")
    except ImportError:
        print(f"Instalando {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
        print(f"OK - {package} instalado")

OK - langchain ja instalado
OK - langgraph ja instalado
OK - requests ja instalado


In [15]:
# Imports
import re
import json
import requests
from collections import defaultdict
from datetime import datetime, timedelta
from typing import Dict, List, Tuple, Optional, Any
from enum import Enum

print("OK - Todos os imports realizados!")

OK - Todos os imports realizados!


In [16]:
# Configuracao de LLM
import os

# Tentar Ollama primeiro (gratuito)
OLLAMA_URL = "http://localhost:11434/api/generate"
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
OPENROUTER_KEY = os.getenv("OPENROUTER_API_KEY", "")

def test_ollama():
    try:
        response = requests.post(OLLAMA_URL, json={
            "model": "mistral",
            "prompt": "test",
            "stream": False
        }, timeout=5)
        return response.status_code == 200
    except:
        return False

def test_openrouter():
    if not OPENROUTER_KEY:
        return False
    try:
        response = requests.get(OPENROUTER_URL, headers={
            "Authorization": f"Bearer {OPENROUTER_KEY}"
        }, timeout=5)
        return response.status_code in [200, 401]
    except:
        return False

# Testar qual LLM esta disponivel
ollama_available = test_ollama()
openrouter_available = test_openrouter()

print(f"\nOllama disponivel: {'SIM' if ollama_available else 'NAO'}")
print(f"OpenRouter disponivel: {'SIM' if openrouter_available else 'NAO'}")

if not ollama_available and not openrouter_available:
    print("\nAVISO: Nenhum LLM disponivel!")
    print("Instale Ollama ou configure OPENROUTER_API_KEY")
else:
    llm_provider = "ollama" if ollama_available else "openrouter"
    print(f"\nUsando: {llm_provider.upper()}")


Ollama disponivel: NAO
OpenRouter disponivel: NAO

AVISO: Nenhum LLM disponivel!
Instale Ollama ou configure OPENROUTER_API_KEY


## PARTE 1: GUARDRAILS (5 TIPOS)

In [17]:
# Guardrail 1: Input Validation
class InputValidation:
    def __init__(self, max_length: int = 1000):
        self.max_length = max_length
        self.sql_patterns = [
            r"\b(DROP|DELETE|INSERT|UPDATE|SELECT)\b",
            r"['\"];.*--",
            r"\bOR\b.*=.*",
        ]
        self.xss_patterns = [
            r"<script[^>]*>.*?</script>",
            r"javascript:",
        ]
    
    def validate(self, query: str) -> Tuple[bool, str]:
        if len(query) > self.max_length:
            return False, "Entrada muito longa"
        if len(query) == 0:
            return False, "Entrada vazia"
        
        for pattern in self.sql_patterns:
            if re.search(pattern, query, re.IGNORECASE):
                return False, "SQL injection detectado"
        
        for pattern in self.xss_patterns:
            if re.search(pattern, query, re.IGNORECASE):
                return False, "XSS detectado"
        
        return True, "OK"

# Guardrail 2: Rate Limiting
class RateLimiter:
    def __init__(self, max_requests: int = 10, window_seconds: int = 60):
        self.max_requests = max_requests
        self.window = timedelta(seconds=window_seconds)
        self.requests = defaultdict(list)
    
    def is_allowed(self, user_id: str) -> Tuple[bool, str]:
        now = datetime.now()
        self.requests[user_id] = [
            t for t in self.requests[user_id]
            if now - t < self.window
        ]
        
        if len(self.requests[user_id]) >= self.max_requests:
            return False, f"Limite atingido ({self.max_requests} requisicoes/min)"
        
        self.requests[user_id].append(now)
        return True, "OK"

# Guardrail 3: PII Detection
class PIIDetection:
    def __init__(self):
        self.patterns = {
            'cpf': r'\d{3}\.\d{3}\.\d{3}-\d{2}',
            'email': r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}',
            'telefone': r'\(\d{2}\)\s?\d{4,5}-\d{4}',
        }
    
    def detect(self, texto: str) -> Dict[str, List[str]]:
        encontrados = {}
        for tipo, pattern in self.patterns.items():
            matches = re.findall(pattern, texto)
            if matches:
                encontrados[tipo] = matches
        return encontrados
    
    def has_pii(self, texto: str) -> bool:
        return len(self.detect(texto)) > 0
    
    def remove(self, texto: str) -> str:
        texto_limpo = texto
        for tipo, pattern in self.patterns.items():
            texto_limpo = re.sub(pattern, f"[{tipo.upper()}_REMOVIDO]", texto_limpo)
        return texto_limpo

# Guardrail 4: Output Filtering
class OutputFiltering:
    def __init__(self, max_length: int = 5000):
        self.max_length = max_length
        self.pii_detector = PIIDetection()
    
    def filter(self, texto: str) -> Tuple[str, Dict]:
        stats = {'pii_removed': 0, 'truncated': False}
        
        pii_found = self.pii_detector.detect(texto)
        if pii_found:
            stats['pii_removed'] = sum(len(v) for v in pii_found.values())
            texto = self.pii_detector.remove(texto)
        
        if len(texto) > self.max_length:
            texto = texto[:self.max_length] + "..."
            stats['truncated'] = True
        
        return texto, stats

# Guardrail 5: Toxicity Filtering
class ToxicityFiltering:
    def __init__(self, threshold: float = 0.5):
        self.threshold = threshold
        self.toxic_keywords = ['idiota', 'burro', 'imbecil', 'xingamento']
    
    def is_toxic(self, texto: str) -> Tuple[bool, float]:
        score = 0.0
        texto_lower = texto.lower()
        
        for keyword in self.toxic_keywords:
            if keyword in texto_lower:
                score += 0.3
        
        score = min(score, 1.0)
        return score >= self.threshold, score

print("OK - 5 Guardrails implementados!")

OK - 5 Guardrails implementados!


## PARTE 2: LANGGRAPH (GRAFO COM NODES E CONDITIONAL EDGES)

In [18]:
# Simular LangGraph (sem dependencia externa)
class SimpleGraph:
    def __init__(self):
        self.nodes = {}
        self.edges = {}
        self.conditional_edges = {}
        self.entry_point = None
    
    def add_node(self, name: str, func):
        self.nodes[name] = func
    
    def add_edge(self, from_node: str, to_node: str):
        if from_node not in self.edges:
            self.edges[from_node] = []
        self.edges[from_node].append(to_node)
    
    def add_conditional_edges(self, from_node: str, decision_func, routes: Dict[str, str]):
        self.conditional_edges[from_node] = (decision_func, routes)
    
    def set_entry_point(self, node: str):
        self.entry_point = node
    
    def invoke(self, state: Dict) -> Dict:
        # CORRECAO: Inicializar estado com todas as chaves necessarias
        state.setdefault('input', '')
        state.setdefault('supervisor_decision', '')
        state.setdefault('agent_1_result', '')
        state.setdefault('agent_2_result', '')
        state.setdefault('agent_3_result', '')
        state.setdefault('final_response', '')
        
        current_node = self.entry_point
        visited = []
        
        while current_node and current_node != "END":
            visited.append(current_node)
            
            # Executar node
            if current_node in self.nodes:
                state = self.nodes[current_node](state)
            
            # Decidir proximo node
            if current_node in self.conditional_edges:
                decision_func, routes = self.conditional_edges[current_node]
                decision = decision_func(state)
                current_node = routes.get(decision, "END")
            elif current_node in self.edges:
                current_node = self.edges[current_node][0] if self.edges[current_node] else "END"
            else:
                current_node = "END"
        
        state['_visited_nodes'] = visited
        return state

print("OK - SimpleGraph (LangGraph simulado) implementado!")


OK - SimpleGraph (LangGraph simulado) implementado!


In [19]:
# Criar LLM wrapper (Ollama ou OpenRouter)
class LLMClient:
    def __init__(self):
        self.ollama_available = test_ollama()
        self.openrouter_available = test_openrouter()
    
    def generate(self, prompt: str, max_tokens: int = 200) -> str:
        if self.ollama_available:
            return self._ollama_generate(prompt, max_tokens)
        elif self.openrouter_available:
            return self._openrouter_generate(prompt, max_tokens)
        else:
            return "[LLM nao disponivel]"
    
    def _ollama_generate(self, prompt: str, max_tokens: int) -> str:
        try:
            response = requests.post(OLLAMA_URL, json={
                "model": "mistral",
                "prompt": prompt,
                "stream": False,
                "num_predict": max_tokens
            }, timeout=30)
            if response.status_code == 200:
                return response.json().get('response', '').strip()
        except:
            pass
        return "[Ollama error]"
    
    def _openrouter_generate(self, prompt: str, max_tokens: int) -> str:
        try:
            response = requests.post(OPENROUTER_URL, headers={
                "Authorization": f"Bearer {OPENROUTER_KEY}",
                "Content-Type": "application/json"
            }, json={
                "model": "mistralai/mistral-7b-instruct",
                "messages": [{"role": "user", "content": prompt}],
                "max_tokens": max_tokens
            }, timeout=30)
            if response.status_code == 200:
                return response.json()['choices'][0]['message']['content'].strip()
        except:
            pass
        return "[OpenRouter error]"

llm = LLMClient()
print("OK - LLM Client criado!")

OK - LLM Client criado!


## PARTE 3: MULTI-AGENTES COM LANGCHAIN (SUPERVISOR + 3 AGENTES)

In [20]:
# Definir State para LangGraph
class AgentState:
    def __init__(self):
        self.data = {
            'input': '',
            'supervisor_decision': '',
            'agent_1_result': '',
            'agent_2_result': '',
            'agent_3_result': '',
            'final_response': ''
        }
    
    def __getitem__(self, key):
        return self.data[key]
    
    def __setitem__(self, key, value):
        self.data[key] = value
    
    def __repr__(self):
        return str(self.data)

# Supervisor (LLM decide qual agente chamar)
def supervisor_node(state: Dict) -> Dict:
    entrada = state['input']
    
    prompt = f"""
    Entrada: {entrada}
    
    Voce eh um supervisor que coordena 3 agentes:
    - Agent 1: Especialista em ANALISE
    - Agent 2: Especialista em PROCESSAMENTO
    - Agent 3: Especialista em RESPOSTA
    
    Qual agente voce vai chamar primeiro?
    Responda APENAS com: agent_1, agent_2 ou agent_3
    """
    
    response = llm.generate(prompt, max_tokens=10)
    
    if "agent_1" in response.lower():
        state['supervisor_decision'] = "agent_1"
    elif "agent_2" in response.lower():
        state['supervisor_decision'] = "agent_2"
    else:
        state['supervisor_decision'] = "agent_3"
    
    print(f"[SUPERVISOR] Decisao: {state['supervisor_decision']}")
    return state

# Agent 1: Analise
def agent_1_node(state: Dict) -> Dict:
    entrada = state['input']
    
    prompt = f"""
    Entrada: {entrada}
    
    Voce eh especialista em ANALISE.
    Analise a entrada e identifique:
    1. Tema principal
    2. Palavras-chave
    3. Tipo de pergunta
    
    Responda em 1-2 linhas.
    """
    
    response = llm.generate(prompt, max_tokens=100)
    state['agent_1_result'] = response
    
    print(f"[AGENT 1 - ANALISE] {response[:80]}...")
    return state

# Agent 2: Processamento
def agent_2_node(state: Dict) -> Dict:
    analise = state.get('agent_1_result', '') or state['input']
    
    prompt = f"""
    Analise anterior: {analise}
    
    Voce eh especialista em PROCESSAMENTO.
    Processe a analise e:
    1. Normalize os dados
    2. Identifique padroes
    3. Prepare para resposta
    
    Responda em 1-2 linhas.
    """
    
    response = llm.generate(prompt, max_tokens=100)
    state['agent_2_result'] = response
    
    print(f"[AGENT 2 - PROCESSAMENTO] {response[:80]}...")
    return state

# Agent 3: Resposta
def agent_3_node(state: Dict) -> Dict:
    processamento = state.get('agent_2_result') or state.get('agent_1_result') or state.get('input', '')
    
    prompt = f"""
    Processamento anterior: {processamento}
    
    Voce eh especialista em RESPOSTA.
    Gere uma resposta clara e concisa:
    1. Responda a pergunta original
    2. Use dados do processamento
    3. Seja breve (1-2 linhas)
    """
    
    response = llm.generate(prompt, max_tokens=100)
    state['agent_3_result'] = response
    state['final_response'] = response
    
    print(f"[AGENT 3 - RESPOSTA] {response[:80]}...")
    return state

# Funcao para decidir proximo node
def decide_next_node(state: Dict) -> str:
    decision = state['supervisor_decision']
    return decision

print("OK - 3 Agentes com LLM REAL implementados!")



OK - 3 Agentes com LLM REAL implementados!


## PARTE 4: INTEGRACAO COMPLETA (GUARDRAILS + LANGGRAPH + MULTI-AGENTES)

In [21]:
# Criar grafo completo
def create_complete_graph():
    graph = SimpleGraph()
    
    # Adicionar nodes
    graph.add_node("supervisor", supervisor_node)
    graph.add_node("agent_1", agent_1_node)
    graph.add_node("agent_2", agent_2_node)
    graph.add_node("agent_3", agent_3_node)
    
    # Adicionar conditional edges (supervisor decide)
    graph.add_conditional_edges(
        "supervisor",
        decide_next_node,
        {
            "agent_1": "agent_1",
            "agent_2": "agent_2",
            "agent_3": "agent_3"
        }
    )
    
    # Adicionar edges normais
    graph.add_edge("agent_1", "agent_3")
    graph.add_edge("agent_2", "agent_3")
    
    # Definir entry point
    graph.set_entry_point("supervisor")
    
    return graph

# Criar pipeline com guardrails
class E6Pipeline:
    def __init__(self):
        self.input_validator = InputValidation()
        self.rate_limiter = RateLimiter(max_requests=10, window_seconds=60)
        self.pii_detector = PIIDetection()
        self.output_filter = OutputFiltering()
        self.toxicity_filter = ToxicityFiltering()
        self.graph = create_complete_graph()
    
    def process(self, user_id: str, query: str) -> Dict:
        result = {
            'success': False,
            'response': None,
            'logs': [],
            'blocked_at': None
        }
        
        # 1. Input Validation
        is_valid, msg = self.input_validator.validate(query)
        result['logs'].append(f"[1] Input Validation: {msg}")
        if not is_valid:
            result['blocked_at'] = 'input_validation'
            return result
        
        # 2. Rate Limiting
        is_allowed, msg = self.rate_limiter.is_allowed(user_id)
        result['logs'].append(f"[2] Rate Limiting: {msg}")
        if not is_allowed:
            result['blocked_at'] = 'rate_limiting'
            return result
        
        # 3. PII Detection (entrada)
        if self.pii_detector.has_pii(query):
            result['logs'].append(f"[3] PII Detection: Bloqueado")
            result['blocked_at'] = 'pii_detection_input'
            return result
        result['logs'].append(f"[3] PII Detection: OK")
        
        # 4. Executar grafo (LangGraph + Multi-Agentes)
        state = {'input': query}
        state = self.graph.invoke(state)
        result['logs'].append(f"[4] LangGraph: Nodes visitados: {state.get('_visited_nodes', [])}")
        
        response = state.get('final_response', 'Sem resposta')
        
        # 5. Output Filtering
        filtered_response, filter_stats = self.output_filter.filter(response)
        result['logs'].append(f"[5] Output Filtering: PII removidos: {filter_stats['pii_removed']}")
        
        # 6. Toxicity Filtering
        is_toxic, score = self.toxicity_filter.is_toxic(filtered_response)
        result['logs'].append(f"[6] Toxicity Filtering: Score {score:.2f}")
        if is_toxic:
            result['blocked_at'] = 'toxicity_filtering'
            return result
        
        # Sucesso!
        result['success'] = True
        result['response'] = filtered_response
        
        return result

pipeline = E6Pipeline()
print("OK - Pipeline completo criado!")

OK - Pipeline completo criado!


## PARTE 5: TESTES COMPLETOS

In [22]:
# Teste 1: Entrada valida com agentes REAIS
print("\n" + "="*80)
print("TESTE 1: ENTRADA VALIDA COM AGENTES REAIS")
print("="*80)

query = "Qual eh a capital do Brasil?"
result = pipeline.process("user1", query)

print(f"\nQuery: {query}")
print(f"Sucesso: {result['success']}")
print(f"Resposta: {result['response']}")
print(f"\nLogs:")
for log in result['logs']:
    print(f"  {log}")


TESTE 1: ENTRADA VALIDA COM AGENTES REAIS
[SUPERVISOR] Decisao: agent_3
[AGENT 3 - RESPOSTA] [LLM nao disponivel]...

Query: Qual eh a capital do Brasil?
Sucesso: True
Resposta: [LLM nao disponivel]

Logs:
  [1] Input Validation: OK
  [2] Rate Limiting: OK
  [3] PII Detection: OK
  [4] LangGraph: Nodes visitados: ['supervisor', 'agent_3']
  [5] Output Filtering: PII removidos: 0
  [6] Toxicity Filtering: Score 0.00


In [23]:
# Teste 2: SQL Injection bloqueado
print("\n" + "="*80)
print("TESTE 2: SQL INJECTION BLOQUEADO")
print("="*80)

query = "'; DROP TABLE users; --"
result = pipeline.process("user2", query)

print(f"\nQuery: {query}")
print(f"Sucesso: {result['success']}")
print(f"Bloqueado em: {result['blocked_at']}")
print(f"\nLogs:")
for log in result['logs']:
    print(f"  {log}")


TESTE 2: SQL INJECTION BLOQUEADO

Query: '; DROP TABLE users; --
Sucesso: False
Bloqueado em: input_validation

Logs:
  [1] Input Validation: SQL injection detectado


In [24]:
# Teste 3: PII detectado
print("\n" + "="*80)
print("TESTE 3: PII DETECTADO")
print("="*80)

query = "Meu CPF eh 123.456.789-00"
result = pipeline.process("user3", query)

print(f"\nQuery: {query}")
print(f"Sucesso: {result['success']}")
print(f"Bloqueado em: {result['blocked_at']}")
print(f"\nLogs:")
for log in result['logs']:
    print(f"  {log}")


TESTE 3: PII DETECTADO

Query: Meu CPF eh 123.456.789-00
Sucesso: False
Bloqueado em: pii_detection_input

Logs:
  [1] Input Validation: OK
  [2] Rate Limiting: OK
  [3] PII Detection: Bloqueado


In [25]:
# Teste 4: Multiplas requisicoes (rate limiting)
print("\n" + "="*80)
print("TESTE 4: RATE LIMITING")
print("="*80)

user_id = "user_rate_test"
for i in range(12):
    result = pipeline.process(user_id, f"Pergunta {i+1}?")
    status = "OK" if result['success'] else "BLOQUEADO"
    print(f"Requisicao {i+1}: {status}")
    if not result['success']:
        print(f"  Bloqueado em: {result['blocked_at']}")


TESTE 4: RATE LIMITING
[SUPERVISOR] Decisao: agent_3
[AGENT 3 - RESPOSTA] [LLM nao disponivel]...
Requisicao 1: OK
[SUPERVISOR] Decisao: agent_3
[AGENT 3 - RESPOSTA] [LLM nao disponivel]...
Requisicao 2: OK
[SUPERVISOR] Decisao: agent_3
[AGENT 3 - RESPOSTA] [LLM nao disponivel]...
Requisicao 3: OK
[SUPERVISOR] Decisao: agent_3
[AGENT 3 - RESPOSTA] [LLM nao disponivel]...
Requisicao 4: OK
[SUPERVISOR] Decisao: agent_3
[AGENT 3 - RESPOSTA] [LLM nao disponivel]...
Requisicao 5: OK
[SUPERVISOR] Decisao: agent_3
[AGENT 3 - RESPOSTA] [LLM nao disponivel]...
Requisicao 6: OK
[SUPERVISOR] Decisao: agent_3
[AGENT 3 - RESPOSTA] [LLM nao disponivel]...
Requisicao 7: OK
[SUPERVISOR] Decisao: agent_3
[AGENT 3 - RESPOSTA] [LLM nao disponivel]...
Requisicao 8: OK
[SUPERVISOR] Decisao: agent_3
[AGENT 3 - RESPOSTA] [LLM nao disponivel]...
Requisicao 9: OK
[SUPERVISOR] Decisao: agent_3
[AGENT 3 - RESPOSTA] [LLM nao disponivel]...
Requisicao 10: OK
Requisicao 11: BLOQUEADO
  Bloqueado em: rate_limiting


## PARTE 6: RESUMO E CONCLUSOES

In [26]:
print("\n" + "="*80)
print("RESUMO: E6 - GUARDRAILS + LANGGRAPH + MULTI-AGENTES")
print("="*80)

summary = """
OK - GUARDRAILS (5 TIPOS)
   1. Input Validation: Bloqueia SQL injection, XSS, comandos perigosos
   2. Rate Limiting: Limita requisicoes por usuario/tempo
   3. PII Detection: Detecta dados pessoais (LGPD)
   4. Output Filtering: Remove PII da resposta
   5. Toxicity Filtering: Detecta linguagem toxica

OK - LANGGRAPH (GRAFO COM NODES E CONDITIONAL EDGES)
   - Supervisor node: LLM decide qual agente chamar
   - Conditional edges: Fluxo inteligente (nao deterministico)
   - State compartilhado: Dados passados entre nodes
   - Nodes visitados: Rastreamento completo

OK - MULTI-AGENTES COM LANGCHAIN (AGENTES REAIS COM LLM)
   - Agent 1 (Analise): LLM especialista em analise
   - Agent 2 (Processamento): LLM especialista em processamento
   - Agent 3 (Resposta): LLM especialista em resposta
   - Supervisor: LLM coordena os agentes
   - LLM REAL: Ollama (local) ou OpenRouter (fallback)

OK - INTEGRACAO COMPLETA
   Fluxo: Entrada Ã¢â€ â€™ Guardrails Ã¢â€ â€™ LangGraph Ã¢â€ â€™ Multi-Agentes Ã¢â€ â€™ Guardrails Ã¢â€ â€™ Saida
   
   Teste 1: Entrada valida processada com sucesso
   Teste 2: SQL injection bloqueado
   Teste 3: PII detectado e bloqueado
   Teste 4: Rate limiting funcionando

PROXIMOS PASSOS:
   1. Explorar codigo dos agentes
   2. Modificar prompts dos agentes
   3. Adicionar mais guardrails
   4. Integrar com FastAPI para API REST
   5. Deploy em producao
"""

print(summary)


RESUMO: E6 - GUARDRAILS + LANGGRAPH + MULTI-AGENTES

OK - GUARDRAILS (5 TIPOS)
   1. Input Validation: Bloqueia SQL injection, XSS, comandos perigosos
   2. Rate Limiting: Limita requisicoes por usuario/tempo
   3. PII Detection: Detecta dados pessoais (LGPD)
   4. Output Filtering: Remove PII da resposta
   5. Toxicity Filtering: Detecta linguagem toxica

OK - LANGGRAPH (GRAFO COM NODES E CONDITIONAL EDGES)
   - Supervisor node: LLM decide qual agente chamar
   - Conditional edges: Fluxo inteligente (nao deterministico)
   - State compartilhado: Dados passados entre nodes
   - Nodes visitados: Rastreamento completo

OK - MULTI-AGENTES COM LANGCHAIN (AGENTES REAIS COM LLM)
   - Agent 1 (Analise): LLM especialista em analise
   - Agent 2 (Processamento): LLM especialista em processamento
   - Agent 3 (Resposta): LLM especialista em resposta
   - Supervisor: LLM coordena os agentes
   - LLM REAL: Ollama (local) ou OpenRouter (fallback)

OK - INTEGRACAO COMPLETA
   Fluxo: Entrada Ã¢â€ â€